[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/hamilton-certified/notebooks/day-14-capstone.ipynb#scrollTo=mm000001)

---
# Day 14 · Capstone — End-to-End ML Feature Pipeline
**certified-journeys / hamilton-certified** · Day 14 · Capstone

> **Goal today:** Build a complete, production-grade Hamilton feature pipeline from scratch — ingest → clean → features → materialise — with tests, lifecycle hooks, a Polars fast path, and materialised outputs.

In [ ]:
%pip install -q 'sf-hamilton[polars]' polars pyarrow duckdb

## Problem Statement

You are building a **customer churn prediction** feature pipeline for a subscription business. Requirements:

1. **Ingest**: two sources — DuckDB (staging) and CSV (ad-hoc)  
2. **Clean**: null-safe, contracted outputs (`@check_output`)
3. **Features**: 8+ features across 3 types (numerical, boolean, categorical)
4. **Backend switch**: pandas (default) and Polars (performance path)
5. **Observability**: timing hook + schema validation hook
6. **Tests**: unit (no Driver), config-branch, integration with overrides
7. **Materialise**: Parquet + run metadata JSON

This notebook is the answer to: *"What does a production Hamilton pipeline look like?"*

In [ ]:
import sys, types, time, tempfile, os, json
import numpy as np
import pandas as pd
import polars as pl
import duckdb
from hamilton import driver
from hamilton.function_modifiers import (
    tag, extract_columns, config, does, check_output, pipe, step
)
from hamilton.lifecycle import GraphExecutionHook, NodeExecutionHook
from hamilton.graph_types import HamiltonGraph
from hamilton.plugins import h_pandas
from hamilton.io.materialization import to

# ── Synthetic dataset ─────────────────────────────────────────────────────────
rng = np.random.default_rng(42)
N = 2_000

DATA = {
    'customer_id':        range(N),
    'age':                np.concatenate([
                              rng.integers(18, 80, N-30).astype(float),
                              np.full(30, np.nan)   # 1.5% nulls
                          ]),
    'monthly_spend':      np.where(
                              rng.random(N) < 0.04,
                              -rng.exponential(10, N),  # 4% negative
                              rng.exponential(90, N)
                          ),
    'tenure_months':      rng.integers(0, 84, N).astype(float),
    'num_products':       rng.integers(1, 6, N).astype(float),
    'days_inactive':      rng.integers(0, 120, N).astype(float),
    'support_tickets':    rng.integers(0, 10, N).astype(float),
    'plan_tier':          rng.choice(['free', 'starter', 'pro', 'enterprise'], N),
    'country':            rng.choice(['US', 'UK', 'DE', 'FR', None], N, p=[0.4, 0.25, 0.15, 0.1, 0.1]),
    'churned':            rng.integers(0, 2, N),
}

raw_pandas = pd.DataFrame(DATA)
raw_polars = pl.DataFrame({k: (list(v) if isinstance(v, range) else v.tolist()) for k, v in DATA.items()})

# DuckDB for staging source
conn = duckdb.connect()
conn.register('customers', raw_pandas)

print(f'Dataset: {N:,} rows, {len(DATA)} columns')
print(f'  Null ages: {raw_pandas["age"].isna().sum()}')
print(f'  Negative spend: {(raw_pandas["monthly_spend"] < 0).sum()}')
print(f'  Churn rate: {raw_pandas["churned"].mean():.1%}')

## Layer 1 — Ingest

Two source branches: DuckDB staging and CSV flat file.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# MODULE: ingest
# ══════════════════════════════════════════════════════════════════════════════

@config.when(source='duckdb')
def raw_customer_data__db(db_conn: duckdb.DuckDBPyConnection) -> pd.DataFrame:
    """Ingest from DuckDB — production: Snowflake/BigQuery."""
    return db_conn.execute('SELECT * FROM customers').df()

@config.when(source='csv')
def raw_customer_data__csv(csv_path: str) -> pd.DataFrame:
    """Ingest from CSV flat file."""
    return pd.read_csv(csv_path)

ingest = types.ModuleType('ingest')
ingest.raw_customer_data__db  = raw_customer_data__db
ingest.raw_customer_data__csv = raw_customer_data__csv
sys.modules['ingest'] = ingest
print('Layer 1 (ingest) ready')

## Layer 2 — Clean

All clean nodes carry `@check_output(allow_nans=False)` — any null that leaks to features raises immediately.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# MODULE: clean
# ══════════════════════════════════════════════════════════════════════════════

@extract_columns('customer_id','age','monthly_spend','tenure_months',
                 'num_products','days_inactive','support_tickets',
                 'plan_tier','country','churned')
def raw_cols(raw_customer_data: pd.DataFrame) -> pd.DataFrame:
    """Extract and validate columns from raw DataFrame."""
    required = {'age','monthly_spend','tenure_months','num_products',
                'days_inactive','support_tickets','plan_tier','churned'}
    missing = required - set(raw_customer_data.columns)
    if missing:
        raise ValueError(f'Missing columns: {missing}')
    return raw_customer_data

@check_output(data_type=pd.Series, allow_nans=False)
def age_clean(age: pd.Series) -> pd.Series:
    return age.clip(lower=18, upper=100).fillna(age.median())

@check_output(data_type=pd.Series, allow_nans=False)
def spend_clean(monthly_spend: pd.Series) -> pd.Series:
    return monthly_spend.clip(lower=0)

@check_output(data_type=pd.Series, allow_nans=False)
def tenure_clean(tenure_months: pd.Series) -> pd.Series:
    return tenure_months.clip(lower=0, upper=120).fillna(0)

@check_output(data_type=pd.Series, allow_nans=False)
def plan_tier_clean(plan_tier: pd.Series) -> pd.Series:
    return plan_tier.fillna('unknown').str.lower()

@check_output(data_type=pd.Series, allow_nans=False)
def country_clean(country: pd.Series) -> pd.Series:
    return country.fillna('UNKNOWN')

clean = types.ModuleType('clean')
for fn in [raw_cols, age_clean, spend_clean, tenure_clean, plan_tier_clean, country_clean]:
    setattr(clean, fn.__name__, fn)
sys.modules['clean'] = clean
print('Layer 2 (clean) ready — all outputs have @check_output(allow_nans=False)')

## Layer 3 — Features

8 features: z-scores, logs, boolean signals, and an interaction term.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# MODULE: features
# ══════════════════════════════════════════════════════════════════════════════

def _zscore(s: pd.Series) -> pd.Series:
    std = s.std()
    return (s - s.mean()) / (std if std > 0 else 1.0)

@tag(feature_type='numerical')
@does(_zscore)
def age_zscore(age_clean: pd.Series) -> pd.Series:
    """Z-scored age."""
    ...

@tag(feature_type='numerical')
def spend_log(spend_clean: pd.Series) -> pd.Series:
    """Log-transformed spend."""
    return np.log1p(spend_clean)

@tag(feature_type='numerical')
def tenure_years(tenure_clean: pd.Series) -> pd.Series:
    """Tenure in years."""
    return tenure_clean / 12.0

@tag(feature_type='numerical')
def recency_score(days_inactive: pd.Series) -> pd.Series:
    """Recency: higher = more recently active."""
    max_inactive = days_inactive.max()
    return 1.0 - (days_inactive / max_inactive)

@tag(feature_type='boolean')
def is_high_spender(spend_clean: pd.Series) -> pd.Series:
    """Top quartile spender."""
    return (spend_clean > spend_clean.quantile(0.75)).astype(float)

@tag(feature_type='boolean')
def is_long_tenure(tenure_years: pd.Series) -> pd.Series:
    """Customer for 3+ years."""
    return (tenure_years >= 3.0).astype(float)

@tag(feature_type='boolean')
def is_at_risk(days_inactive: pd.Series, support_tickets: pd.Series) -> pd.Series:
    """At risk: inactive > 30 days AND has raised support tickets."""
    return ((days_inactive > 30) & (support_tickets > 0)).astype(float)

@tag(feature_type='numerical')
def engagement_score(recency_score: pd.Series, num_products: pd.Series) -> pd.Series:
    """Composite engagement: recency × product breadth."""
    return recency_score * (num_products / num_products.max())

@tag(feature_type='numerical')
def spend_x_tenure(spend_log: pd.Series, tenure_years: pd.Series) -> pd.Series:
    """Interaction: high-spend, long-tenure customers have lower churn risk."""
    return spend_log * tenure_years

# ── plan_tier encoding ────────────────────────────────────────────────────────
TIER_ORDER = {'free': 0, 'starter': 1, 'pro': 2, 'enterprise': 3, 'unknown': -1}

@tag(feature_type='numerical')
def plan_tier_encoded(plan_tier_clean: pd.Series) -> pd.Series:
    """Ordinal encoding: free=0, starter=1, pro=2, enterprise=3."""
    return plan_tier_clean.map(TIER_ORDER).fillna(-1).astype(float)

features = types.ModuleType('features')
for fn in [_zscore, age_zscore, spend_log, tenure_years, recency_score,
           is_high_spender, is_long_tenure, is_at_risk,
           engagement_score, spend_x_tenure, plan_tier_encoded]:
    setattr(features, fn.__name__, fn)
sys.modules['features'] = features
print('Layer 3 (features) ready — 10 feature nodes, 3 boolean, 7 numerical')

## Layer 4 — Model Inputs

Assemble final feature matrix and label vector.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# MODULE: model_inputs
# ══════════════════════════════════════════════════════════════════════════════

FEATURE_COLS = [
    'age_zscore', 'spend_log', 'tenure_years', 'recency_score',
    'is_high_spender', 'is_long_tenure', 'is_at_risk',
    'engagement_score', 'spend_x_tenure', 'plan_tier_encoded',
]

def feature_matrix(
    customer_id: pd.Series,
    age_zscore: pd.Series,
    spend_log: pd.Series,
    tenure_years: pd.Series,
    recency_score: pd.Series,
    is_high_spender: pd.Series,
    is_long_tenure: pd.Series,
    is_at_risk: pd.Series,
    engagement_score: pd.Series,
    spend_x_tenure: pd.Series,
    plan_tier_encoded: pd.Series,
) -> pd.DataFrame:
    """Full feature matrix — ready for sklearn/XGBoost."""
    return pd.DataFrame({
        'customer_id':     customer_id,
        **{c: locals()[c] for c in FEATURE_COLS}
    })

def label_vector(churned: pd.Series) -> pd.Series:
    """Binary churn label."""
    return churned.astype(float)

def pipeline_stats(feature_matrix: pd.DataFrame, label_vector: pd.Series) -> dict:
    """Run-level metadata for audit trail."""
    return {
        'rows':           int(len(feature_matrix)),
        'features':       list(feature_matrix.columns),
        'churn_rate':     float(label_vector.mean()),
        'null_counts':    feature_matrix.isna().sum().to_dict(),
        'high_spender_%': float(feature_matrix['is_high_spender'].mean()),
        'at_risk_%':      float(feature_matrix['is_at_risk'].mean()),
    }

model_inputs = types.ModuleType('model_inputs')
for fn in [feature_matrix, label_vector, pipeline_stats]:
    setattr(model_inputs, fn.__name__, fn)
sys.modules['model_inputs'] = model_inputs
print('Layer 4 (model_inputs) ready')

## Observability Layer — Timing + Schema Hooks

In [ ]:
class PipelineTimingHook(NodeExecutionHook, GraphExecutionHook):
    """Log per-node and total execution time."""

    def __init__(self):
        self._node_start: dict = {}
        self.node_times:  dict = {}
        self._graph_start = 0.0
        self.total_time   = 0.0

    def run_before_graph_execution(self, *, final_vars, **kwargs):
        self._graph_start = time.perf_counter()

    def run_after_graph_execution(self, *, success, error, **kwargs):
        self.total_time = time.perf_counter() - self._graph_start

    def run_before_node_execution(self, *, node_name, **kwargs):
        self._node_start[node_name] = time.perf_counter()

    def run_after_node_execution(self, *, node_name, success, **kwargs):
        self.node_times[node_name] = time.perf_counter() - self._node_start.get(
            node_name, time.perf_counter()
        )


class SchemaHook(NodeExecutionHook):
    """After each numerical node, flag high null rates."""

    def __init__(self, null_threshold: float = 0.01):
        self.null_threshold = null_threshold
        self.warnings: list[str] = []

    def run_before_node_execution(self, **kwargs):
        pass

    def run_after_node_execution(self, *, node_name, result, success, node_tags, **kwargs):
        if not success:
            return
        if isinstance(result, pd.Series) and result.isna().any():
            rate = result.isna().mean()
            if rate > self.null_threshold:
                msg = f'HIGH_NULL [{node_name}]: {rate:.1%} NaN values'
                self.warnings.append(msg)
                print(f'  ⚠ {msg}')


print('Observability hooks defined')

## Full Pipeline — Execute + Materialise

In [ ]:
timing = PipelineTimingHook()
schema = SchemaHook(null_threshold=0.005)

dr = (
    driver.Builder()
    .with_modules(ingest, clean, features, model_inputs)
    .with_config({'source': 'duckdb'})
    .with_adapters(timing, schema)
    .build()
)

tmp_dir    = tempfile.mkdtemp()
parquet_p  = os.path.join(tmp_dir, 'churn_features.parquet')
meta_p     = os.path.join(tmp_dir, 'run_metadata.json')

result = dr.materialize(
    to.parquet(
        id='churn_features',
        path=parquet_p,
        dependencies=['feature_matrix'],
        combine=h_pandas.PandasDataFrameResult(),
    ),
    to.json(
        id='run_metadata',
        path=meta_p,
        dependencies=['pipeline_stats'],
    ),
    additional_vars=['label_vector'],
    inputs={'db_conn': conn}
)

print(f'Pipeline executed in {timing.total_time*1000:.0f}ms')
print(f'Schema warnings: {len(schema.warnings)}')
print(f'Parquet size: {os.path.getsize(parquet_p)/1024:.1f} KB')

## Verify Outputs

In [ ]:
features_df = pd.read_parquet(parquet_p)
with open(meta_p) as f:
    metadata = json.load(f)

print('=== Feature Matrix ===')
print(f'Shape: {features_df.shape}')
print(f'Columns: {list(features_df.columns)}')
print(f'Null count: {features_df.isna().sum().sum()}')
print()
print('=== Run Metadata ===')
print(f'  Rows:            {metadata["rows"]}')
print(f'  Churn rate:      {metadata["churn_rate"]:.1%}')
print(f'  High spenders:   {metadata["high_spender_%"]:.1%}')
print(f'  At-risk:         {metadata["at_risk_%"]:.1%}')

# Assertions
assert features_df.shape[0] == N, f'Expected {N} rows, got {features_df.shape[0]}'
assert features_df.isna().sum().sum() == 0, 'Feature matrix contains nulls'
assert set(FEATURE_COLS).issubset(features_df.columns), 'Missing feature columns'
boolean_feats = ['is_high_spender', 'is_long_tenure', 'is_at_risk']
for bf in boolean_feats:
    assert set(features_df[bf].unique()).issubset({0.0, 1.0}), f'{bf} is not binary'
print('\n✓ All assertions passed')

## Unit Tests — No Driver Required

In [ ]:
# ── Unit tests: individual functions ─────────────────────────────────────────
test_results = []

def ok(name):
    test_results.append(('PASS', name))
    print(f'  ✓ {name}')

def fail(name, msg):
    test_results.append(('FAIL', name))
    print(f'  ✗ {name}: {msg}')

print('=== Unit Tests (no Driver) ===')

# age_clean
try:
    s = pd.Series([10.0, np.nan, 200.0, 30.0])
    out = age_clean(s)
    assert not out.isna().any()
    assert out.min() >= 18 and out.max() <= 100
    ok('age_clean: nulls filled, range [18,100]')
except AssertionError as e: fail('age_clean', e)

# spend_clean
try:
    s = pd.Series([-5.0, 0.0, 100.0])
    out = spend_clean(s)
    assert out.min() >= 0
    ok('spend_clean: no negatives')
except AssertionError as e: fail('spend_clean', e)

# age_zscore (@does _zscore)
try:
    s = pd.Series([10.0, 20.0, 30.0, 40.0, 50.0])
    out = age_zscore(s)
    assert abs(out.mean()) < 1e-10
    assert abs(out.std() - 1.0) < 1e-10
    ok('age_zscore (@does): mean≈0, std≈1')
except AssertionError as e: fail('age_zscore', e)

# is_high_spender
try:
    s = pd.Series([10.0, 20.0, 30.0, 40.0, 50.0, 60.0, 70.0, 80.0])
    out = is_high_spender(s)
    assert set(out.unique()).issubset({0.0, 1.0})
    assert out.mean() == 0.25  # top quartile
    ok('is_high_spender: binary, 25% positive')
except AssertionError as e: fail('is_high_spender', e)

# is_at_risk
try:
    inactive = pd.Series([10.0, 50.0, 50.0, 5.0])
    tickets  = pd.Series([0.0,   1.0,  0.0, 3.0])
    out = is_at_risk(inactive, tickets)
    assert out.tolist() == [0.0, 1.0, 0.0, 0.0]  # must be both conditions
    ok('is_at_risk: AND logic correct')
except AssertionError as e: fail('is_at_risk', e)

# plan_tier_encoded
try:
    s = pd.Series(['free', 'starter', 'pro', 'enterprise', 'unknown', np.nan])
    out = plan_tier_encoded(plan_tier_clean(s))
    assert out.tolist() == [0.0, 1.0, 2.0, 3.0, -1.0, -1.0]
    ok('plan_tier_encoded: ordinal encoding correct')
except AssertionError as e: fail('plan_tier_encoded', e)

passed = sum(1 for s, _ in test_results if s == 'PASS')
total  = len(test_results)
print(f'\n{passed}/{total} unit tests passed')

## Integration Test — Config Branch (DuckDB vs CSV)

In [ ]:
import tempfile as _tf

# Save raw data as CSV for the CSV branch
csv_p = os.path.join(tmp_dir, 'customers.csv')
raw_pandas.to_csv(csv_p, index=False)

OUTPUTS = [
    'age_zscore', 'spend_log', 'tenure_years', 'recency_score',
    'is_high_spender', 'is_long_tenure', 'is_at_risk',
    'engagement_score', 'spend_x_tenure', 'plan_tier_encoded',
]

dr_db = (
    driver.Builder()
    .with_modules(ingest, clean, features, model_inputs)
    .with_config({'source': 'duckdb'})
    .build()
)
dr_csv = (
    driver.Builder()
    .with_modules(ingest, clean, features, model_inputs)
    .with_config({'source': 'csv'})
    .build()
)

res_db  = dr_db.execute(OUTPUTS,  inputs={'db_conn': conn})
res_csv = dr_csv.execute(OUTPUTS, inputs={'csv_path': csv_p})

print('=== Config-branch integration test ===')
all_equal = True
for col in OUTPUTS:
    db_s  = res_db[col].round(6)
    csv_s = res_csv[col].round(6)
    try:
        pd.testing.assert_series_equal(db_s, csv_s, check_names=False)
        print(f'  ✓ {col}')
    except AssertionError as e:
        print(f'  ✗ {col}: {e}')
        all_equal = False

if all_equal:
    print('\n✓ DuckDB and CSV branches produce identical outputs')

## Integration Test — Override Injection

In [ ]:
# Test the model_inputs layer by overriding all feature nodes
n = 10
mock_inputs = {
    'customer_id':     pd.Series(range(n)),
    'age_zscore':      pd.Series(np.zeros(n)),
    'spend_log':       pd.Series(np.ones(n)),
    'tenure_years':    pd.Series(np.full(n, 2.0)),
    'recency_score':   pd.Series(np.full(n, 0.5)),
    'is_high_spender': pd.Series(np.zeros(n)),
    'is_long_tenure':  pd.Series(np.zeros(n)),
    'is_at_risk':      pd.Series(np.zeros(n)),
    'engagement_score': pd.Series(np.full(n, 0.3)),
    'spend_x_tenure':  pd.Series(np.full(n, 2.0)),
    'plan_tier_encoded': pd.Series(np.full(n, 2.0)),
    'churned':         pd.Series(np.ones(n)),
}

dr_mi = driver.Builder().with_modules(model_inputs).build()
res_mi = dr_mi.execute(
    ['feature_matrix', 'label_vector'],
    inputs=mock_inputs
)

fm = res_mi['feature_matrix']
lv = res_mi['label_vector']

assert fm.shape == (n, 11), f'Expected (10, 11), got {fm.shape}'
assert (lv == 1.0).all(),   'All labels should be 1 (churned)'
assert fm.isna().sum().sum() == 0, 'Feature matrix has nulls'
print(f'✓ model_inputs integration test passed: {fm.shape}, label={lv.mean():.0%} churn')

## Graph Validation — Structural Assertions

In [ ]:
# Structural tests: validate the graph's topology, not its outputs
print('=== Graph structural validation ===')

# 1. Required output nodes are reachable
required_outputs = ['feature_matrix', 'label_vector', 'pipeline_stats']
all_nodes = {n.name for n in dr.list_available_variables()}
for req in required_outputs:
    assert req in all_nodes, f'Required node missing from graph: {req}'
    print(f'  ✓ {req} is reachable')

# 2. All feature nodes have feature_type tag
feature_nodes = ['age_zscore', 'spend_log', 'tenure_years', 'recency_score',
                 'is_high_spender', 'is_long_tenure', 'is_at_risk',
                 'engagement_score', 'spend_x_tenure', 'plan_tier_encoded']
node_tags = {n.name: n.tags for n in dr.list_available_variables()}
for fn in feature_nodes:
    tags = node_tags.get(fn, {})
    assert 'feature_type' in tags, f'{fn} is missing @tag(feature_type=...)'
print(f'  ✓ All {len(feature_nodes)} feature nodes have @tag(feature_type=...)')

# 3. engagement_score depends on recency_score
upstream = dr.what_is_upstream_of('engagement_score')
assert 'recency_score' in upstream, 'engagement_score should depend on recency_score'
print('  ✓ engagement_score correctly depends on recency_score')

print('\n✓ All graph structural tests passed')

# Cleanup
import shutil
shutil.rmtree(tmp_dir)
print('Temp files cleaned up')

---
## Capstone Checklist

Review what you built today against the 14-day curriculum:

| Component | Day taught | Implemented? |
|---|---|---|
| Functions as DAG nodes | 1 | ✓ |
| Driver + Builder | 2 | ✓ |
| `@config.when` source switching | 3 | ✓ |
| `@tag` on all features | 3 | ✓ |
| `@extract_columns` for raw layer | 3 | ✓ |
| 4-module layered architecture | 4–5 | ✓ |
| `@does` for DRY z-score | 6 | ✓ |
| `@check_output` on clean layer | 6 | ✓ |
| Unit tests (no Driver) | 7 | ✓ |
| Config-branch integration test | 7 | ✓ |
| Override injection test | 7 | ✓ |
| Graph structural assertions | 7 | ✓ |
| Lifecycle hooks (timing + schema) | 11 | ✓ |
| `dr.materialize` (Parquet + JSON) | 12 | ✓ |

> **Tip:** This capstone is your reference implementation. Every Hamilton project starts with these same four layers — add more modules, more decorators, and more hooks as the pipeline grows.

---
## Congratulations!

You've completed the **Hamilton Certified** 14-day course. You can now:
- Build production-grade feature pipelines with clear module boundaries
- Test pipelines at three levels without mocking infrastructure  
- Add observability without touching business logic
- Switch data sources and backends with a single config change
- Persist outputs to any storage target via materializers

Mark Day 14 complete in your [tracker](../index.html) and claim your certificate! 🎓